### 텐서 생성, 기본 연산 (해야함)

### requires_grad 의미 파악

In [1]:
import torch

In [5]:
tensor1 = torch.tensor(1., requires_grad=True)
tensor2 = torch.tensor(2., requires_grad=True)
tensor3 = torch.tensor(3., requires_grad=True)
output_tensor = torch.mul(tensor1, tensor2)
output_tensor = torch.mul(output_tensor, tensor3)
output_tensor.backward()

print(f'tensor1.grad: {tensor1.grad}')
print(f'tensor2.grad: {tensor2.grad}')
print(f'tensor3.grad: {tensor3.grad}')
print(f'output_tensor.grad: {output_tensor.grad}')

tensor1.grad: 6.0
tensor2.grad: 3.0
tensor3.grad: 2.0
output_tensor.grad: None


/tmp/ipykernel_8938/963793391.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'output_tensor.grad: {output_tensor.grad}')


In [8]:
tensor1 = torch.tensor(1., requires_grad=False)
tensor2 = torch.tensor(2., requires_grad=True)
tensor3 = torch.tensor(3., requires_grad=False)
output_tensor = torch.mul(tensor1, tensor2)
output_tensor = torch.mul(output_tensor, tensor3)
output_tensor.backward()

print(f'tensor1.grad: {tensor1.grad}')
print(f'tensor2.grad: {tensor2.grad}')
print(f'tensor3.grad: {tensor3.grad}')
print(f'output_tensor.grad: {output_tensor.grad}')

tensor1.grad: None
tensor2.grad: 3.0
tensor3.grad: None
output_tensor.grad: None


/tmp/ipykernel_8938/239290478.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'output_tensor.grad: {output_tensor.grad}')


backward() Function를 통해서 역전파를 시켜주게 되면 Tensor 의 grad 에 값이 할당

requires_grad=false 인 경우 grad 값이 None으로 고정 True 인 경우 자동 미분

- [참고1](https://westlife0615.tistory.com/866)
- [참고2](https://nuguziii.github.io/dev/dev-003/)


### Autograd 체험

In [18]:
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
z = x1**2 + x2**3 + x1*x2

print(f"함수 z = x1^2 + x2^3 + x1*x2")
print(f"x1={x1.item()}, x2={x2.item()}, z={z.item()}")

z.backward()
print(f"Autograd = {x1.grad} (이론값: 2*x1 + x2 = {2*2 + 3})")
print(f"Autograd = {x2.grad} (이론값: 3*x2^2 + x1 = {3*3**2 + 2})")

함수 z = x1^2 + x2^3 + x1*x2
x1=2.0, x2=3.0, z=37.0
Autograd = 7.0 (이론값: 2*x1 + x2 = 7)
Autograd = 29.0 (이론값: 3*x2^2 + x1 = 29)


### 모델 설계

In [19]:
import torch.nn as nn

In [20]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP,self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=30, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.layer3 = nn.Sequential(
            nn.Linear(in_features=30*5*5, out_features=10, bias=True),
            nn.ReLU(inplace=True)
        )

        def forward(self, x):
            x = self.layer1(x)
            x = self.layer2(x)
            x = x.view(x.shape[0], -1)
            x = self.layer3(x)
            return x

model = MLP()

In [21]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n전체 파라미터 수: {total_params}")
print(f"학습 가능한 파라미터 수: {trainable_params}")


전체 파라미터 수: 60404
학습 가능한 파라미터 수: 60404


### GPU 활용

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device: ',device)

device:  cuda


In [24]:
tensor_cpu = torch.randn(3, 3)
tensor_gpu = tensor_cpu.to(device)
print('tensor_cpu: ',tensor_cpu.device)
print('tensor_gpu: ',tensor_gpu.device)

model = model.to(device)
print('model dvice: ',next(model.parameters()).device)

tensor_cpu:  cpu
tensor_gpu:  cuda:0
model dvice:  cuda:0


### 기본 훈련 루프 (해야함)

(1)예측

(2)손실 계산

(3)그래디언트 초기화

(4)역전파

(5)가중치 업데이트